In [ ]:
import cv2
import numpy as np
from keras.models import load_model
import tensorflow as tf
import keras.backend as K
from PIL import Image
from tensorflow.keras.losses import sparse_categorical_crossentropy, binary_crossentropy        


def calc_cm_vals(y_true, y_pred, class_index=None):
    true = tf.reshape(y_true, (-1,))
    
    # pred = tf.argmax(y_pred, axis=1) #softmax
    
    # Convert probabilities to binary predictions using the threshold
    threshold = 0.5
    pred = tf.cast(tf.greater_equal(y_pred, threshold), tf.int32)
    pred = tf.reshape(pred, (-1,))
    
    tp = tf.reduce_sum(tf.cast(tf.logical_and(tf.equal(true, class_index), 
                                              tf.equal(pred, class_index)), 
                                               tf.float32))
    fp = tf.reduce_sum(tf.cast(tf.logical_and(tf.not_equal(true, class_index), 
                                              tf.equal(pred, class_index)), 
                                               tf.float32))
    fn = tf.reduce_sum(tf.cast(tf.logical_and(tf.equal(true, class_index), 
                                             tf.not_equal(pred, class_index)), 
                                              tf.float32))
    return tp, fp, fn

def recall_0(y_true, y_pred, class_index=0):
    tp, fp, fn = calc_cm_vals(y_true, y_pred, class_index)
    return tp / (tp + fn + K.epsilon()) 

def precision_0(y_true, y_pred, class_index=0):
    tp, fp, fn = calc_cm_vals(y_true, y_pred, class_index)
    return tp / (tp + fp + K.epsilon())  
    
def precision_1(y_true, y_pred, class_index=1):
    tp, fp, fn = calc_cm_vals(y_true, y_pred, class_index)
    return tp / (tp + fp + K.epsilon()) 

def recall_1(y_true, y_pred, class_index=1):
    tp, fp, fn = calc_cm_vals(y_true, y_pred, class_index)
    return tp / (tp + fn + K.epsilon()) 

def f1_score_1(y_true, y_pred, class_index=1):
    precision = precision_1(y_true, y_pred, class_index)
    recall = recall_1(y_true, y_pred, class_index)
    f1_score = 2 * (precision * recall) / (precision + recall + K.epsilon())
    return f1_score

def f1_score_1_loss(y_true, y_pred):
    f1_score = f1_score_1(y_true, y_pred)
    ce_loss = sparse_categorical_crossentropy(y_true, y_pred, 
                                              from_logits=False, axis=-1)
    alpha = 0.00001
    combined_loss = alpha * ce_loss + (1 - alpha) * (1-f1_score)
    return combined_loss 
    
def precision_0_loss(y_true, y_pred):
    precision = precision_0(y_true, y_pred)  
    ce_loss = sparse_categorical_crossentropy(y_true, y_pred, 
                                        from_logits=False, axis=-1)
    alpha = 0.001
    combined_loss = alpha * ce_loss + (1 - alpha) * (1-precision)
    return combined_loss 

def precision_1_loss(y_true, y_pred):
    precision = precision_1(y_true, y_pred)  
    ce_loss = sparse_categorical_crossentropy(y_true, y_pred, 
                                              from_logits=False, axis=-1)
    alpha = 0.001
    combined_loss = alpha * ce_loss + (1 - alpha) * (1-precision)
    return combined_loss 
    
def recall_1_loss(y_true, y_pred):
    recall = recall_1(y_true, y_pred)
    ce_loss = sparse_categorical_crossentropy(y_true, y_pred, 
                                              from_logits=False, axis=-1)
    alpha = 0.001
    combined_loss = alpha * ce_loss + (1 - alpha) * (1-recall)
    return combined_loss 

def preprocess_img(image, target_size):
    if image is not None:
        height, width, channels = image.shape
        if width<height :
            image = cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE)                
    preprocessed_image = cv2.resize(image, target_size)
    reshaped_image = preprocessed_image.reshape(1, *preprocessed_image.shape)

    image = reshaped_image.astype(np.float32)
    image = image / 255.0
    return image

def predict_image(preprocessed_img, model, label_map, 
                  activation='sigmoid', thresh=0.5, verbose=0):
    prediction = model.predict(preprocessed_img, verbose=verbose)
    
    if activation=='sigmoid':
        prediction = np.array(prediction)
        threshold = thresh
        prediction = (prediction >= threshold).astype(int)
        prediction = prediction[0]

    elif activation=='softmax':
        prediction = np.array([np.argmax(pred) for pred in prediction])  
    
    reverse_label_map = {idx: img_type for img_type, idx in label_map.items()}
    predicted_class = reverse_label_map[prediction[0]] 
    return predicted_class





# def main(img_path = ''):
#     model_path = r"D:\New folder\Fish-Data-Science-project-Jupyter-notebooks\New Experiments\fish_freshness_classification - Copy\saved_models\Sardine\freshness_saved_models\2011_1_densenet_single_dropout_0.2_recall_1_precision_0_accuracy_mobilenetv2_model.h5"
#     label_map = {'Bad': 0, 'Good': 1}
#     target_size = (280, 180)
#     activation='sigmoid'
#     thresh=0.5
#     custom_metrics = {'recall_1':recall_1, 'precision_1':precision_1, 
#                       'recall_0':recall_0, 'precision_0':precision_0,
#                       'precision_0_loss':precision_0_loss,
#                       'precision_1_loss':precision_1_loss}
    
#     model = load_model(model_path, custom_objects=custom_metrics)
#     # model = load_model(model_path)
#     # Open an image file
#     img = Image.open(img_path) 
#     img = np.array(img)
#     preprocessed_img = preprocess_img(img, target_size)
#     predicted_class = predict_image(preprocessed_img, model, label_map,
                                    # activation, thresh, verbose=1)
#     # print(predicted_class)
#     return predicted_class

In [ ]:

custom_metrics = {'recall_1':recall_1, 'precision_1':precision_1, 
                    'recall_0':recall_0, 'precision_0':precision_0}

model_path = r"Models\Single-group-all\v2 mnv2_224_rgb_sigmoid_5e_adam_default_lr_0.1val_bcel.h5"

model = load_model(model_path, custom_objects=custom_metrics)

def test(img_path = ''):
    label_map = {'Group': 0, 'Single': 1}
    target_size = (224, 224)
    activation='sigmoid'
    thresh=0.5

    img = Image.open(img_path) 
    img = np.array(img)
    preprocessed_img = preprocess_img(img, target_size)
    predicted_class = predict_image(preprocessed_img, model, label_map,
                                    activation, thresh, verbose=0)
    # print(predicted_class)
    return predicted_class

In [3]:
# test(r"C:\Users\sowmy\Downloads\QzenseLabs\qZense Dataset\S3 Daily Data\2023-05-25\Roopchand\Good\20230525123906365_roopchand_good.jpeg")

In [4]:
import os
from tqdm import tqdm
import shutil


s3_daily_data_folder = r"C:\Users\sowmy\Downloads\QzenseLabs\qZense Dataset\S3 Daily Data"
# s3_daily_data_folder = r"C:\Users\sowmy\Downloads\QzenseLabs\qZense Dataset\Manual Data"

def get_dates_list(start_date=None, end_date=None):
    if start_date is None:
        dates_list = os.listdir(s3_daily_data_folder)
    else:
        if end_date is None:
            dates_list = [start_date]
        else:
            dates_list = []
            for date in sorted(os.listdir(s3_daily_data_folder)):
                if not (start_date <= date <= end_date): continue
                dates_list.append(date)
    return dates_list


def classify_s3_daily_data(dates_list):
    for date in dates_list:
        date_folder = os.path.join(s3_daily_data_folder, date)
        for species in tqdm(os.listdir(date_folder), desc = f"{date}"):
            species_folder = os.path.join(date_folder, species)
            if not os.path.isdir(species_folder): continue
            for type in os.listdir(species_folder):
                type_folder = os.path.join(species_folder, type)
                if not os.path.isdir(type_folder): continue
                for img in sorted(os.listdir(type_folder)):
                    img_path = os.path.join(type_folder, img)
                    if not os.path.isfile(img_path): continue
                    # print(img_path)
                    grp_img_path = os.path.join(type_folder, 'Group', img)
                    single_img_path = os.path.join(type_folder, 'Single', img) 
                    if os.path.exists(grp_img_path) or os.path.exists(single_img_path):
                        os.remove(img_path)
                        continue
                    
                    try:    
                        predicted_class = test(img_path)
                        new_folder = os.path.join(type_folder, predicted_class)                        
                        os.makedirs(new_folder, exist_ok=True)
                        shutil.move(img_path, new_folder)
                    except:
                        continue


In [6]:
# date format YYYY-MM-DD
dates_list = get_dates_list(start_date='2024-06-04', end_date='2024-06-10')
# dates_list = get_dates_list(start_date='2023-05-20')
# dates_list = get_dates_list()

classify_s3_daily_data(dates_list)

2024-06-04: 100%|██████████| 1/1 [00:00<?, ?it/s]
